# Chapter 4

Add your content here.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# 1. THE DATASET
# The task: Input [x], Target [x + 2]. 
# However, the relationship is implicit.
# We want the model to "think" about the intermediate step (x+1).
def get_data(batch_size=1):
    x = torch.randint(1, 10, (batch_size, 1)).float()
    y_target = x + 2.0 
    return x, y_target

# 2. THE MODEL
class ThinkingModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Main network: Just tries to guess output from input
        self.main_head = nn.Linear(1, 1)
        
        # Thought Generator: Generates a 'thought' value from input
        # In a real LLM, this would generate text tokens. Here, just a number.
        self.thought_generator = nn.Sequential(
            nn.Linear(1, 4),
            nn.ReLU(),
            nn.Linear(4, 1) # Outputs mean of a Normal distribution
        )
        
        # Mixing Head: Takes Input AND Thought to make final prediction
        self.mixed_head = nn.Linear(2, 1) 

    def forward(self, x):
        # Path 1: Standard Prediction (Baseline)
        pred_no_thought = self.main_head(x)
        
        # Path 2: Thinking Process
        thought_mean = self.thought_generator(x)
        
        # Stochasticity: We must SAMPLE the thought to learn via REINFORCE
        # In real LLMs, we sample tokens. Here we sample a number.
        std_dev = 0.5
        noise = torch.randn_like(thought_mean) * std_dev
        thought_value = thought_mean + noise
        
        # Path 3: Prediction WITH thought
        # We concatenate input x and the generated thought
        combined_input = torch.cat([x, thought_value], dim=1)
        pred_with_thought = self.mixed_head(combined_input)
        
        return pred_no_thought, pred_with_thought, thought_mean, thought_value

# 3. TRAINING LOOP (The Quiet-STaR Logic)
model = ThinkingModel()
optimizer = optim.Adam(model.parameters(), lr=0.01)
baseline_moving_avg = 0.0

print("Training internal thoughts...")

for step in range(500):
    optimizer.zero_grad()
    
    # Get Data
    x, y_true = get_data()
    
    # Forward Pass
    pred_base, pred_thought, mean, sampled_thought = model(x)
    
    # --- CALCULATE REWARD ---
    # Loss is Mean Squared Error (closer is better)
    loss_base = (pred_base - y_true)**2
    loss_thought = (pred_thought - y_true)**2
    
    # Reward: How much did the thought IMPROVE the error?
    # If loss_thought < loss_base, Reward is positive.
    reward = (loss_base.detach() - loss_thought.detach())
    
    # --- REINFORCE LOSS ---
    # Maximize Reward => Minimize -Reward
    # J ~ (R - b) * log_prob(action)
    # For Gaussian distribution, log_prob is related to -(x-mean)^2
    
    log_prob = -0.5 * ((sampled_thought - mean) / 0.5)**2
    
    # Update baseline (simple exponential moving average)
    baseline_moving_avg = 0.9 * baseline_moving_avg + 0.1 * reward.mean().item()
    advantage = reward - baseline_moving_avg
    
    # The Magic Formula: Loss = - (Advantage * Log_Prob)
    # This trains the thought_generator to pick better thoughts
    policy_loss = -(advantage * log_prob).mean()
    
    # We also want the final prediction to be accurate (Supervised Loss)
    final_prediction_loss = loss_thought.mean()
    
    total_loss = final_prediction_loss + policy_loss
    total_loss.backward()
    optimizer.step()
    
    if step % 100 == 0:
        print(f"Step {step}: Reward={reward.mean().item():.4f}, Thought Used={sampled_thought.item():.2f} for Input={x.item()}")

print("\nExplanation: If the Reward is positive, the model learns that the sampled 'Thought' value was useful for solving x + 2.")

Training internal thoughts...
Step 0: Reward=24.4272, Thought Used=-1.98 for Input=3.0
Step 100: Reward=130.4056, Thought Used=5.01 for Input=6.0
Step 200: Reward=67.5596, Thought Used=3.80 for Input=4.0
Step 300: Reward=130.2201, Thought Used=6.11 for Input=6.0
Step 400: Reward=11.8459, Thought Used=2.82 for Input=1.0

Explanation: If the Reward is positive, the model learns that the sampled 'Thought' value was useful for solving x + 2.
